# Week 20: MLOps - CI/CD, Monitoring, and LLM Observability

## Where we are

Last week you turned the fraud classifier into a production asset: a SageMaker training job, an MLflow run, a model registered in the Model Registry, a deployed endpoint, and a `classify_with_finetuned_model` tool plugged into `week19_supervisor`. That is the artifact. This week we operate it.

## Learning objectives

By the end of this lab you will be able to:

1. Add OpenTelemetry-based tracing to a Strands agent and view per-decision traces in Langfuse without modifying agent code
2. Enable data capture on a SageMaker endpoint and schedule a Model Monitor job
3. Detect statistical drift in pandas between a baseline S3-loaded dataset and a fresh batch
4. Create a CloudWatch alarm on endpoint latency that notifies an SNS topic
5. Explain when to reach for LiteLLM versus a full agent framework

## The mental model

Week 18 measured RAG quality OFFLINE on a fixed eval set with RAGAS. That tells you what the system can do in a lab. Week 20 measures the SAME system ONLINE on live traffic with Langfuse, Model Monitor, and CloudWatch. That tells you what the system is actually doing right now, in production, on real users.

In [ ]:
# Studio Lab already has boto3 and sagemaker. We add Langfuse + Strands OTEL + LiteLLM.
# Use plain !pip - Studio Lab does not support %pip magic.

!pip install -q \
    "sagemaker==2.257.3" \
    "strands-agents>=1.37,<2" \
    "strands-agents-tools[mem0-memory]>=0.2" \
    "langfuse>=2.50" \
    "langchain-aws>=0.2" \
    "litellm>=1.50" \
    "opentelemetry-exporter-otlp>=1.27" \
    "opentelemetry-sdk>=1.27"

import os
import json
import base64
import boto3
import sagemaker
from sagemaker import get_execution_role

print("sagemaker:", sagemaker.__version__)

In [ ]:
# Standard MLOps setup on SageMaker Studio: pick up the execution role and region
# from the Studio environment. No getpass, no dbutils, no IAM keys in the notebook.

sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

# Strands tools read these from env
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

# Clients we will need across the notebook
sagemaker_client = boto3.client("sagemaker", region_name=AWS_REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=AWS_REGION)
cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)
s3 = boto3.client("s3", region_name=AWS_REGION)

# Endpoint, KB, model, bucket constants carried over from Week 19
ENDPOINT_NAME = "fraud-classifier-endpoint"
KB_ID = "FARSQGTONR"
BEDROCK_MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"  # consistent with Weeks 15-19
RERANK_MODEL_ID = "cohere.rerank-v3-5:0"
S3_BUCKET = "bread-academy-week19-shared"

print("AWS region:", AWS_REGION)
print("Role:", role)
print("Endpoint:", ENDPOINT_NAME)
print("Bedrock model:", BEDROCK_MODEL_ID)
print("S3 bucket:", S3_BUCKET)

In [ ]:
# Pre-flight: fail loud now if anything we depend on is missing.

# 1. Endpoint is in service
resp = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
assert resp["EndpointStatus"] == "InService", (
    f"Endpoint {ENDPOINT_NAME} is {resp['EndpointStatus']}. "
    "Ask your instructor to redeploy the Week 19 endpoint."
)
print("Endpoint status:", resp["EndpointStatus"])

# 2. Bedrock LLM responds
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
try:
    bedrock_runtime.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print("Bedrock LLM probe: ok")
except Exception as e:
    print("Ask your instructor to enable Bedrock access for", BEDROCK_MODEL_ID)
    raise

# 3. KB exists
bedrock_agent = boto3.client("bedrock-agent", region_name=AWS_REGION)
bedrock_agent.get_knowledge_base(knowledgeBaseId=KB_ID)
print("Bedrock KB probe: ok")

# 4. S3 fraud CSV reachable
s3.head_object(Bucket=S3_BUCKET, Key="datasets/fraud_transactions.csv")
print("S3 fraud CSV probe: ok")

In [ ]:
# In Week 19's Databricks environment the supervisor came from %run on a helper
# notebook. SageMaker Studio Lab has no %run; we declare the supervisor inline
# so this notebook is self-contained.

import json as _json
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def classify_with_finetuned_model(transaction_description: str) -> str:
    """Fast fraud pre-screen using the facebook/bart-large-mnli zero-shot classifier.

    Uses the pretrained model students downloaded from S3 in the previous cell.
    Returns a JSON string with keys: label ('fraud' or 'legit'), confidence (0-1).
    Call this BEFORE slower LLM-based tools to cheaply filter obvious cases.
    """
    result = classifier(transaction_description, candidate_labels=FRAUD_LABELS_TOOL)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    label = "fraud" if "fraudulent" in top_label else "legit"
    return _json.dumps({
        "label":      label,
        "confidence": round(float(top_score), 4),
    })

# Stub policy retriever tool (Week 18 KB+rerank flow; stubbed here for self-containment)
@tool
def policy_retriever_tool(query: str) -> str:
    """Stub of the Week 18 policy retriever. In real code this hits KB FARSQGTONR + Cohere Rerank."""
    return ("Policy: transactions above $5000 from new merchants require manual review. "
            "Card-not-present with country mismatch is auto-flagged.")

SUPERVISOR_SYSTEM_PROMPT = (
    "You are a fraud triage supervisor. For each transaction: "
    "1) ALWAYS call classify_with_finetuned_model first. "
    "2) If confidence >= 0.85, return its decision with reasoning. "
    "3) If confidence < 0.85, also call policy_retriever_tool and synthesize a final call. "
    "Always include your reasoning."
)

_supervisor_model = BedrockModel(model_id=BEDROCK_MODEL_ID, region_name=AWS_REGION)
week19_supervisor = Agent(
    model=_supervisor_model,
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
    tools=[classify_with_finetuned_model, policy_retriever_tool],
)

print(type(week19_supervisor).__name__)
print("Tools registered on supervisor:", week19_supervisor.tool_names)

In [ ]:
import os
import tarfile as _tarfile
from transformers import pipeline as hf_pipeline

PRETRAINED_LOCAL_DIR = "/tmp/bart-mnli-pretrained"
PRETRAINED_TARBALL   = "/tmp/bart-mnli-pretrained.tar.gz"

if not os.path.isdir(PRETRAINED_LOCAL_DIR):
    print("Downloading pretrained model from S3 (one-time, ~250 MB)...")
    s3.download_file(S3_BUCKET, "pretrained/model.tar.gz", PRETRAINED_TARBALL)
    os.makedirs(PRETRAINED_LOCAL_DIR, exist_ok=True)
    with _tarfile.open(PRETRAINED_TARBALL, "r:gz") as tar:
        tar.extractall(PRETRAINED_LOCAL_DIR)
    print(f"Model extracted to {PRETRAINED_LOCAL_DIR}")
else:
    print(f"Model already cached at {PRETRAINED_LOCAL_DIR}")

classifier = hf_pipeline(
    "zero-shot-classification",
    model=PRETRAINED_LOCAL_DIR,
    device=-1,
)

FRAUD_LABELS_TOOL = ["fraudulent transaction", "legitimate transaction"]

_test = classifier(
    "High-value wire transfer to unverified offshore account at 3am",
    candidate_labels=FRAUD_LABELS_TOOL,
)
print(f"Sanity check: {_test['labels'][0]} ({_test['scores'][0]:.3f})")

## Part 1 - Langfuse observability (online traces)

### The problem

You ship `week19_supervisor`. A user complains it gave a wrong fraud decision on transaction `T-99812` at 14:32 UTC yesterday. You need to answer:

- Which tools did the supervisor call, in what order?
- What did `classify_with_finetuned_model` return?
- Which policy chunks did `policy_retriever_tool` retrieve, and what was their rerank score?
- How long did the whole decision take, and what did it cost in tokens?

Without tracing this is impossible. RAGAS does not help, because RAGAS is offline. You need a per-request flight recorder. That is Langfuse.

### How Strands + Langfuse fit together

Strands emits OpenTelemetry spans for every model call and tool call. Langfuse speaks OTLP. We point Strands at Langfuse with five lines, then re-run the agent. No code change to the agent itself.

The magic is in `StrandsTelemetry().setup_otlp_exporter()`. After that call, every `agent(prompt)` invocation produces a trace in the Langfuse UI.

In [ ]:
# Five-line wiring. After this cell every supervisor call shows up in Langfuse.
#
# Langfuse keys come from environment variables set by the instructor in the
# Studio Lab user config. If they are not in env, fall back to AWS Secrets
# Manager under name "bread-academy/langfuse".

from strands.telemetry import StrandsTelemetry

LANGFUSE_PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = os.environ.get("LANGFUSE_HOST")

if not (LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY and LANGFUSE_HOST):
    print("Env vars not set - falling back to Secrets Manager.")
    sm = boto3.client("secretsmanager", region_name=AWS_REGION)
    secret = json.loads(sm.get_secret_value(SecretId="bread-academy/langfuse")["SecretString"])
    LANGFUSE_PUBLIC_KEY = secret["public_key"]
    LANGFUSE_SECRET_KEY = secret["secret_key"]
    LANGFUSE_HOST = secret["host"]

# Langfuse OTLP endpoint expects basic auth with base64(public:secret)
LANGFUSE_AUTH = base64.b64encode(
    f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()
).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = LANGFUSE_HOST + "/api/public/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"
# Optional: a service name groups traces in the UI
os.environ["OTEL_SERVICE_NAME"] = "bread-academy-week20"

StrandsTelemetry().setup_otlp_exporter()
print("Langfuse OTLP exporter wired. Host:", LANGFUSE_HOST)

In [ ]:
# Run a known transaction through the unchanged week19_supervisor.
# Then open Langfuse and find the trace.

sample_prompt = (
    "Investigate transaction T-10042 for customer C-7781. "
    "Use the fine-tuned classifier, check policy, and recommend an action."
)

response = week19_supervisor(sample_prompt)
print(str(response.message)[:500])

# Force-flush the OTel BatchSpanProcessor so the span reaches Langfuse before
# we move on. Without this, short demo runs can exit before the batch exporter
# ships its buffer.
from opentelemetry import trace as _otel_trace
_otel_trace.get_tracer_provider().force_flush()

print()
print("Open", LANGFUSE_HOST, "-> Traces. The trace will show:")
print("- the top-level supervisor span")
print("- each tool invocation as a child span")
print("- the bedrock-runtime converse calls with token counts")

### Lab 1 - Trace five transactions and find the slowest one (15 min)

You are the on-call data scientist. Run the supervisor on five different fraud transactions, then open Langfuse and answer three questions:

1. Which transaction took the longest end-to-end?
2. Which tool dominated the latency for that transaction?
3. What were the total input and output tokens for the slowest one?

You do NOT need to change `week19_supervisor`. You only need to invoke it five times and inspect the Langfuse UI.

Hints:
- Use a `for` loop over `test_transactions` already defined for you below.
- Write descriptive prompts; Strands sets the trace name from the prompt automatically.
- The Langfuse UI is at `LANGFUSE_HOST`. Sort by Latency descending in the Traces table.

### Stretch

Add a custom attribute `transaction_id` to each trace using OpenTelemetry's current span. Look up `opentelemetry.trace.get_current_span().set_attribute(...)`.

### Homework extension

Pipe the production supervisor through Langfuse for 24 hours of real traffic (simulated by replaying the fraud CSV from S3). Build a small pandas dashboard from the Langfuse exported events to track p95 latency by tool.

In [ ]:
# Lab 1 starter. Define the five test transactions, then loop and call
# week19_supervisor. After the loop, open Langfuse to inspect traces.

test_transactions = [
    "T-10042", "T-10117", "T-10298", "T-10355", "T-10401",
]

# YOUR CODE: loop over test_transactions and call week19_supervisor
# with a clear prompt that names the transaction id.
results = None  # YOUR CODE

print("Done. Open", LANGFUSE_HOST, "-> Traces to inspect.")

In [ ]:
# SAFETY-NET for Lab 1. Run this only if you did not finish the lab.
# Skip if results is already populated.

if results is None:
    print("Using Lab 1 safety-net.")
    results = []
    for tx in test_transactions:
        prompt = (
            f"Investigate transaction {tx}. Use the fine-tuned classifier, "
            "check policy with the retriever, and recommend an action."
        )
        out = week19_supervisor(prompt)
        results.append({"tx": tx, "text": str(out.message)[:200]})
    print(f"Ran {len(results)} transactions through the supervisor.")

### Think about it

Langfuse stores every prompt and every tool argument by default. A fraud transaction record contains a customer ID and a dollar amount. Is that PII for your jurisdiction? If it is, what would you change about this setup before pointing it at real production traffic? Consider redaction, self-hosted Langfuse, sampling, and retention windows.

## Part 2 - SageMaker data capture and Model Monitor

### What Langfuse will NOT tell you

Langfuse traces the supervisor's decisions. It does not watch the fine-tuned classifier endpoint at the input-feature level. If next month the distribution of `transaction_amount` shifts because of a new product launch, the endpoint will keep returning predictions and Langfuse will keep showing happy traces, while the classifier silently goes off the rails.

That is what SageMaker Model Monitor is for. It samples requests and responses into S3 (data capture), then a scheduled job compares the captured distribution to a baseline you computed from training data. If a feature drifts beyond a threshold, the monitoring job writes a violation report.

### Two enablement points

Data capture is configured on the EndpointConfig, not the endpoint directly. You can:
1. Set it at endpoint-creation time (cleanest, what we recommend for new endpoints)
2. Update an existing endpoint's config to enable it after the fact (what we will do today, since Week 19 created the endpoint without capture)

In [ ]:
# Enable data capture by creating a new endpoint config that points at the
# same model but adds DataCaptureConfig, then update the endpoint to use it.

import time
from datetime import datetime

capture_prefix = f"fraud-classifier/data-capture/{datetime.utcnow():%Y-%m-%d}"
capture_s3_uri = f"s3://{S3_BUCKET}/{capture_prefix}"

# Find the model name behind the current endpoint
current_cfg_name = sagemaker_client.describe_endpoint(
    EndpointName=ENDPOINT_NAME
)["EndpointConfigName"]
current_cfg = sagemaker_client.describe_endpoint_config(
    EndpointConfigName=current_cfg_name
)
model_name = current_cfg["ProductionVariants"][0]["ModelName"]
variant_name = current_cfg["ProductionVariants"][0]["VariantName"]
instance_type = current_cfg["ProductionVariants"][0]["InstanceType"]

new_cfg_name = f"fraud-classifier-cfg-capture-{int(time.time())}"
sagemaker_client.create_endpoint_config(
    EndpointConfigName=new_cfg_name,
    ProductionVariants=[{
        "VariantName": variant_name,
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": instance_type,
    }],
    DataCaptureConfig={
        "EnableCapture": True,
        "InitialSamplingPercentage": 100,
        "DestinationS3Uri": capture_s3_uri,
        "CaptureOptions": [
            {"CaptureMode": "Input"},
            {"CaptureMode": "Output"},
        ],
    },
)
sagemaker_client.update_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=new_cfg_name,
)
print("Update started. New config:", new_cfg_name)
print("Capture S3:", capture_s3_uri)

In [ ]:
# Wait for the endpoint to come back InService, then invoke a few times.
import time

while True:
    status = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
    if status == "InService":
        break
    print("Endpoint status:", status, "- waiting 30s")
    time.sleep(30)

# Send 10 sample invocations using the zero-shot classification format.
# The endpoint now runs facebook/bart-large-mnli, which expects candidate_labels.
FRAUD_LABELS = ["fraudulent transaction", "legitimate transaction"]
CAT_LABELS   = ["grocery", "gas_station", "online_retail", "restaurant", "wire_transfer",
                 "atm_withdrawal", "subscription", "travel", "luxury_goods", "electronics"]

sample_text = "Card-not-present purchase of $482.50 at electronics merchant, no country mismatch"

for i in range(10):
    sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": sample_text, "parameters": {"candidate_labels": FRAUD_LABELS}}),
    )

# Also demonstrate the two-call pattern for fraud + category classification.
resp = sm_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps({"inputs": sample_text, "parameters": {"candidate_labels": FRAUD_LABELS}}),
)
prediction = json.loads(resp["Body"].read())
top_label = prediction["labels"][0]
top_score = prediction["scores"][0]
print(f"Fraud classification: {top_label} (confidence={top_score:.3f})")

resp2 = sm_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps({"inputs": sample_text, "parameters": {"candidate_labels": CAT_LABELS}}),
)
pred2 = json.loads(resp2["Body"].read())
print(f"Category:             {pred2['labels'][0]} (confidence={pred2['scores'][0]:.3f})")

# Inspect what landed in S3. Capture can take a minute to flush.
time.sleep(60)
listing = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=capture_prefix)
for obj in listing.get("Contents", [])[:5]:
    print(obj["Key"], obj["Size"], "bytes")

### Lab 2 - Baseline + scheduled monitor (15 min)

You will create a Model Monitor baseline from the training data and schedule a monitoring job on the endpoint. The lab is mostly configuration. The point is to understand the moving parts:

1. Baseline dataset (training data, in S3 as CSV)
2. `DefaultModelMonitor.suggest_baseline(...)` job (one-shot, writes constraints.json + statistics.json)
3. `monitor.create_monitoring_schedule(...)` (hourly, watches the data capture path)

You will not wait for the scheduled run to complete in class. You will start it and confirm the schedule shows up under SageMaker -> Monitoring jobs.

### Stretch

Read the generated `constraints.json` from S3 and print the inferred type and completeness for the captured input field.

### Homework extension

Trigger the monitoring job by sending 1000 invocations with transaction narratives that represent a shifted distribution (e.g. all large-amount cross-border transactions), then read the violation report from S3.

In [ ]:
# Lab 2 starter. Use SageMaker SDK to baseline and schedule.

from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator
from sagemaker.model_monitor.dataset_format import DatasetFormat

baseline_input_s3 = f"s3://{S3_BUCKET}/datasets/fraud_transactions.csv"
baseline_results_s3 = f"s3://{S3_BUCKET}/fraud-classifier/monitoring/baseline-results"
schedule_output_s3 = f"s3://{S3_BUCKET}/fraud-classifier/monitoring/schedule-results"

# YOUR CODE: build a DefaultModelMonitor with role=role, instance_count=1,
# instance_type="ml.m5.xlarge", and volume_size_in_gb=20.
monitor = None  # YOUR CODE

# YOUR CODE: call monitor.suggest_baseline(...) with baseline_dataset=baseline_input_s3,
# dataset_format=DatasetFormat.csv(header=True), output_s3_uri=baseline_results_s3,
# wait=True. This runs a Processing job that emits statistics.json + constraints.json.

# YOUR CODE: call monitor.create_monitoring_schedule(...) with:
# - monitor_schedule_name="fraud-classifier-hourly"
# - endpoint_input=ENDPOINT_NAME
# - output_s3_uri=schedule_output_s3
# - statistics=monitor.baseline_statistics()
# - constraints=monitor.suggested_constraints()
# - schedule_cron_expression=CronExpressionGenerator.hourly()
# - enable_cloudwatch_metrics=True

print("Schedule created. Check SageMaker console -> Monitoring jobs.")

In [ ]:
# SAFETY-NET for Lab 2. Run only if monitor is still None.

if monitor is None:
    print("Using Lab 2 safety-net.")
    monitor = DefaultModelMonitor(
        role=role,
        instance_count=1,
        instance_type="ml.m5.xlarge",
        volume_size_in_gb=20,
        max_runtime_in_seconds=1800,
        sagemaker_session=sess,
    )
    monitor.suggest_baseline(
        baseline_dataset=baseline_input_s3,
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=baseline_results_s3,
        wait=True,
    )
    monitor.create_monitoring_schedule(
        monitor_schedule_name="fraud-classifier-hourly",
        endpoint_input=ENDPOINT_NAME,
        output_s3_uri=schedule_output_s3,
        statistics=monitor.baseline_statistics(),
        constraints=monitor.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True,
    )
    print("Schedule created via safety-net.")

## Part 3 - Drift detection with pandas

### Two kinds of drift

- **Data drift** (covariate shift): the input feature distribution changes. Example: average `transaction_amount` jumps from $80 to $260 after a marketing campaign.
- **Concept drift**: the relationship between features and the label changes. Example: a new fraud ring uses small amounts that the old model treats as safe.

Model Monitor catches data drift on whatever the endpoint sees. But often the drift is visible upstream, in your offline data lake, before it ever reaches the endpoint. We will load the training fraud CSV from S3 with pandas, synthesize a drifted "fresh batch" by perturbing it, and compute PSI on the numeric features `amount` and `days_since_last_txn` plus an L1 distance on the categorical feature `merchant_country`. The canonical drift baked into the Week 19 synthetic data: the US share of `merchant_country` drops from 0.80 to 0.55 in days 60-89.

We are deliberately NOT using Spark here. The whole monitoring loop is meant to run on a small Studio Lab notebook, not on a cluster. pandas + numpy is enough; if your batches outgrow pandas you can move the same logic to a SageMaker Processing job later.

In [ ]:
# Pull the original training slice from S3 and simulate a drifted batch.
# In production the "drifted" batch is last week's transactions; here we
# synthesize one so the lesson is visible in class.

import pandas as pd
import numpy as np

# Download the fraud CSV from the shared bucket
local_csv = "/tmp/fraud_transactions.csv"
s3.download_file(S3_BUCKET, "datasets/fraud_transactions.csv", local_csv)

baseline_df = pd.read_csv(local_csv)[
    ["amount", "merchant_category", "merchant_country", "days_since_last_txn", "is_fraud"]
]

# Simulate the canonical Week 19/20 drift: US merchant_country share drops from 0.80 to 0.55,
# fraud rate rises from 3% to 4.5%. This matches the synthetic data generator used in Week 19.
rng = np.random.default_rng(42)
drifted_df = baseline_df.copy()

# Amount drift: shift up 3x with noise to make PSI visible
drifted_df["amount"] = (
    drifted_df["amount"] * 3.0 + rng.uniform(0, 50, size=len(drifted_df))
)

# Merchant country drift: replace some US rows with other countries
# so US share drops from ~0.80 to ~0.55
is_us = drifted_df["merchant_country"] == "US"
flip = is_us & (rng.random(len(drifted_df)) > 0.69)
drifted_df.loc[flip, "merchant_country"] = rng.choice(["GB", "MX", "DE", "CA"], size=flip.sum())

print("Baseline rows:", len(baseline_df))
print("Drifted rows:", len(drifted_df))
print()
print("Baseline US share:", round((baseline_df["merchant_country"] == "US").mean(), 3))
print("Drifted US share: ", round((drifted_df["merchant_country"] == "US").mean(), 3))
print()
print("Baseline amount summary:")
print(baseline_df["amount"].describe().round(2))
print()
print("Drifted amount summary:")
print(drifted_df["amount"].describe().round(2))

In [ ]:
# Population Stability Index (PSI) is the standard in finance for drift.
# We compute it on amount and days_since_last_txn by bucketing both distributions
# into 10 deciles of the baseline and comparing fractional populations.

import math

def psi(baseline_arr, current_arr, bins=10):
    """PSI between two 1-D numeric arrays using baseline-derived decile edges."""
    quantiles = np.percentile(baseline_arr, [100 * i / bins for i in range(1, bins)])
    edges = np.concatenate([[-np.inf], quantiles, [np.inf]])

    b_counts, _ = np.histogram(baseline_arr, bins=edges)
    c_counts, _ = np.histogram(current_arr, bins=edges)
    nb, nc = b_counts.sum(), c_counts.sum()

    score = 0.0
    eps = 1e-10
    for i in range(bins):
        pb = max(b_counts[i] / nb, eps)
        pc = max(c_counts[i] / nc, eps)
        score += (pc - pb) * math.log(pc / pb)
    return score

amount_psi = psi(baseline_df["amount"].values,
                 drifted_df["amount"].values)
days_psi = psi(baseline_df["days_since_last_txn"].values,
               drifted_df["days_since_last_txn"].values)

print(f"PSI amount:               {amount_psi:.3f}")
print(f"PSI days_since_last_txn:  {days_psi:.3f}")
print("Rule of thumb: PSI < 0.1 stable, 0.1-0.25 moderate drift, > 0.25 significant drift.")

### Lab 3 - Detect categorical drift and decide whether to retrain (15 min)

PSI works for numeric features. For categorical features like `merchant_country` we compare category frequency vectors. The Week 19 synthetic data generator bakes in a specific drift: the US share of `merchant_country` drops from roughly 0.80 in the baseline to roughly 0.55 in the drifted batch. Your task:

1. Build a function `category_drift(baseline_df, current_df, column)` that returns the L1 distance between normalized category frequency vectors.
2. Run it on `merchant_country`.
3. Define a simple decision rule: if any numeric feature has PSI > 0.25 OR any categorical feature has L1 distance > 0.3, print "RETRAIN". Otherwise print "OK".

Hints:
- Use `df[column].value_counts(normalize=True)` on each DataFrame to get a Series of frequencies that already sums to 1.
- Align the two Series on the union of category names; missing categories get 0.
- L1 distance = sum of absolute differences.

### Stretch

Wrap the whole drift check in a function and log the drift scores as parameters to a new SageMaker managed MLflow run tagged `drift_check`. That gives you a historical record of drift over time.

### Homework extension

Schedule this drift check as a daily SageMaker Processing job. On RETRAIN, trigger the Week 19 SageMaker training pipeline via the SageMaker SDK so the loop closes automatically. (Week 21 will show you the Airflow version of the same flow.)

In [ ]:
# Lab 3 starter.

def category_drift(baseline, current, column):
    # YOUR CODE
    return None  # YOUR CODE

cat_drift = None  # YOUR CODE

# YOUR CODE
decision = None  # YOUR CODE

print("Categorical drift:", cat_drift)
print("Decision:", decision)

In [ ]:
# SAFETY-NET for Lab 3.

if decision is None:
    print("Using Lab 3 safety-net.")
    def category_drift(baseline, current, column):
        b = baseline[column].value_counts(normalize=True)
        c = current[column].value_counts(normalize=True)
        keys = b.index.union(c.index)
        b = b.reindex(keys, fill_value=0.0)
        c = c.reindex(keys, fill_value=0.0)
        return float((b - c).abs().sum())

    cat_drift = category_drift(baseline_df, drifted_df, "merchant_country")
    decision = "RETRAIN" if (amount_psi > 0.25 or days_psi > 0.25 or cat_drift > 0.3) else "OK"
    print("Categorical drift:", cat_drift)
    print("Decision:", decision)

## Part 4 - CloudWatch alarm on endpoint latency

Model Monitor fires on data quality. Langfuse fires on agent behavior. Neither pages you at 3am when the endpoint just becomes slow.

SageMaker endpoints emit metrics to CloudWatch in the `AWS/SageMaker` namespace: `ModelLatency`, `Invocations`, `Invocation4XXErrors`, `Invocation5XXErrors`. We will create one alarm: if `ModelLatency` average exceeds 1000 ms for 5 consecutive 1-minute periods, send a message to the SNS topic the instructor pre-created. SNS fans out to email, Slack, PagerDuty, whatever.

This part is a demo only. You watch the cell run and the alarm appear in the CloudWatch console.

In [ ]:
# Create a CloudWatch alarm wired to an SNS topic the instructor provisioned.
# SNS topic ARN comes from env var SNS_TOPIC_ARN. If unset, instructor's
# fallback ARN is used (replace the literal with the actual ARN for your cohort).
#
# Note: AWS/SageMaker ModelLatency is published in MICROSECONDS, not milliseconds.
# 1_000_000 microseconds = 1 second of model latency.

SNS_TOPIC_ARN = os.environ.get(
    "SNS_TOPIC_ARN",
    "arn:aws:sns:us-east-1:535146832369:fraud-endpoint-alerts",
)

cloudwatch.put_metric_alarm(
    AlarmName="fraud-classifier-high-latency",
    AlarmDescription="Fires when average ModelLatency exceeds 1s (1_000_000 microseconds) for 5 minutes",
    ActionsEnabled=True,
    MetricName="ModelLatency",
    Namespace="AWS/SageMaker",
    Statistic="Average",
    Dimensions=[
        {"Name": "EndpointName", "Value": ENDPOINT_NAME},
        {"Name": "VariantName", "Value": "AllTraffic"},
    ],
    Period=60,
    EvaluationPeriods=5,
    Threshold=1_000_000.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    AlarmActions=[SNS_TOPIC_ARN],
)

print("Alarm created. Open CloudWatch -> Alarms to see it.")
print("Subscribe yourself to", SNS_TOPIC_ARN, "to receive the page.")

## Part 5 - LiteLLM as a unified interface (brief)

Strands gives you agents. Langfuse gives you traces. But sometimes you do not need an agent; you just need to call an LLM and log the call. LiteLLM is the "boring" version of this pattern: a single Python function `litellm.completion(...)` that speaks to any provider, with built-in callbacks for Langfuse and others.

You will not build with LiteLLM today. You will see it work in one cell. The point is the interface: same call, any provider, free Langfuse logging.

In [ ]:
# Single demo of LiteLLM auto-logging to Langfuse. No agent involved.

import litellm

# Tell LiteLLM to log everything to Langfuse. The Langfuse env vars
# from Part 1 are still set, so it picks them up automatically.
litellm.success_callback = ["langfuse"]
litellm.failure_callback = ["langfuse"]

resp = litellm.completion(
    model="bedrock/us.anthropic.claude-3-haiku-20240307-v1:0",
    messages=[
        {"role": "user", "content": "In one sentence, why is online monitoring required for ML systems?"}
    ],
    max_tokens=80,
    temperature=0,
)
print(resp.choices[0].message.content)

# Flush the OTel exporter once more so this LiteLLM generation reaches Langfuse
# before the notebook ends.
from opentelemetry import trace as _otel_trace
_otel_trace.get_tracer_provider().force_flush()

print()
print("This call appears in Langfuse as a Generation (not a Trace), tagged with the model name.")

### Think about it

You now have four production signals on the same fraud system:

1. Langfuse traces (per-decision, qualitative)
2. SageMaker Model Monitor (feature distribution, statistical)
3. pandas drift jobs (upstream, decision-level)
4. CloudWatch alarms (operational, time-series)

Which of these would have caught the following, in order of speed?
- The endpoint instance ran out of memory at 3:04 am
- A new fraud pattern using sub-dollar amounts started yesterday
- The supervisor started calling `policy_retriever_tool` 14 times per decision because of a prompt regression you shipped Friday
- The training-serving skew on `merchant_country` has been growing for three weeks

There is no single right ordering. The point is that each signal has a job, and none of them substitutes for the others.

## A note on CI/CD

We did not build a GitHub Actions workflow in this notebook. In an AWS-centric MLOps shop you usually combine two pieces:

1. A GitHub Actions workflow that, on push to `main`, runs unit tests, packages the training code, and calls the SageMaker Python SDK to submit the Week 19 training job.
2. A SageMaker Pipeline (or EventBridge schedule) that runs the drift check from Part 3 and triggers (1) if RETRAIN is decided.

The "CI" half is testing your training code before it can run. The "CD" half is the drift check that decides when to run it. You already have both ingredients; the wiring is environment-specific homework.

## Wrap-up

You took a deployed Strands-based fraud system and put four production guardrails on it:

- Langfuse + OTEL traces on `week19_supervisor` with five lines of setup
- SageMaker data capture and a Model Monitor schedule on `fraud-classifier-endpoint`
- A pandas PSI + L1 drift detector on S3-loaded data, with a clear RETRAIN rule
- A CloudWatch alarm on endpoint latency wired to SNS
- A glimpse of LiteLLM as the simpler unified-interface pattern

## Homework

1. Finish the homework extension on Lab 1 (24h replay + p95 dashboard in pandas).
2. Finish the homework extension on Lab 3 (daily SageMaker Processing job that triggers Week 19's training pipeline on RETRAIN).
3. Read the SageMaker Model Monitor docs for the four built-in monitor types: Data Quality, Model Quality, Bias Drift, Feature Attribution Drift. Pick one of the latter three and write a 200-word note on when you would add it to this pipeline.

Next week (Week 21) we move to orchestration with Airflow / MWAA. The drift check from Lab 3 will become a DAG.